# 03. Compress와 Isolate

Compress는 필요한 사실을 짧게 보존하고, Isolate는 질문마다 전달할 근거를 분리합니다.

## 실행 환경 준비

첫 코드 셀은 노트북에서 사용할 패키지 버전을 맞춥니다. 설치가 끝나면 이어지는 셀을 순서대로 실행합니다.

In [1]:
%pip install -q "openai==3.8.0" "python-dotenv==1.2.3" "tiktoken==0.14.0"

Note: you may need to restart the kernel to use updated packages.


## 1. API 연결 준비

In [2]:
from dotenv import find_dotenv, load_dotenv
from openai import OpenAI
import os

env_path = find_dotenv(usecwd=True)
if not env_path:
    raise FileNotFoundError("현재 폴더의 상위 경로에서 .env를 찾지 못했습니다.")
load_dotenv(env_path)
if not os.getenv("OPENAI_API_KEY"):
    raise RuntimeError(".env에 OPENAI_API_KEY가 없습니다.")
client = OpenAI()
MODEL = "gpt-5.4-mini"
print("API 실습 준비 완료")

API 실습 준비 완료


## 2. 사실·조건·예외·출처를 보존합니다

길이만 줄이면 중요한 예외를 잃을 수 있습니다. 네 필드를 모두 확인한 뒤 한 줄로 압축합니다.

In [3]:
def compress(record: dict[str, str]) -> str:
    required = ("fact", "condition", "exception", "source")
    missing = set(required) - record.keys()
    if missing:
        raise ValueError(f"빠진 필드: {sorted(missing)}")
    return " | ".join(f"{key}: {record[key]}" for key in required)

loan = {"fact": "7권을 14일 대출", "condition": "회원 1명 기준",
        "exception": "예약이 없으면 1회 7일 연장", "source": "E01 p.3"}
compressed = compress(loan)
print(compressed)

fact: 7권을 14일 대출 | condition: 회원 1명 기준 | exception: 예약이 없으면 1회 7일 연장 | source: E01 p.3


## 3. 질문별 입력을 분리합니다

대출 질문에는 대출 근거만 넣습니다. 태그는 읽기 구조를 돕지만 보안 경계는 아닙니다.

In [4]:
question = "대출 권수, 기간, 연장 조건을 알려 주세요."
prompt = f"[근거]\n{compressed}\n\n[질문]\n{question}"
print(prompt)

[근거]
fact: 7권을 14일 대출 | condition: 회원 1명 기준 | exception: 예약이 없으면 1회 7일 연장 | source: E01 p.3

[질문]
대출 권수, 기간, 연장 조건을 알려 주세요.


## 4. 압축한 근거로 답을 생성합니다

In [5]:
response = client.responses.create(
    model=MODEL,
    instructions="근거의 사실·조건·예외·출처를 빠뜨리지 마세요.",
    input=prompt,
    max_output_tokens=300,
    reasoning={"effort": "none"},
)
print(response.output_text)
print("input_tokens:", response.usage.input_tokens)
print("output_tokens:", response.usage.output_tokens)

회원 1명 기준으로 **7권을 14일간 대출**할 수 있습니다.  
또한 **예약이 없으면 1회 7일 연장**이 가능합니다.  

- **대출 권수:** 7권  
- **대출 기간:** 14일  
- **연장 조건:** 예약이 없을 때만 1회 7일 연장 가능  

**출처:** E01 p.3
input_tokens: 94
output_tokens: 98


## 5. 일부러 실패해 보기

예외가 빠진 압축 레코드는 생성 단계에서 거부합니다.

In [6]:
broken = {"fact": "7권을 14일 대출", "condition": "회원 1명 기준",
          "source": "E01 p.3"}
try:
    compress(broken)
except ValueError as error:
    print("예상한 실패:", error)
else:
    raise AssertionError("필드 누락 오류가 발생해야 합니다.")

예상한 실패: 빠진 필드: ['exception']


## 정리

압축은 짧게 만드는 작업과 보존해야 할 필드를 검사하는 작업을 함께 수행합니다. 질문별 입력은 호출 전에 분리합니다.